<h1 style="padding-top: 25px;padding-bottom: 25px;text-align: left; padding-left: 10px; background-color: #DDDDDD;
    color: black;"> <img style="float: left; padding-right: 10px;" src="https://raw.githubusercontent.com/Harvard-IACS/2018-CS109A/master/content/styles/iacs.png" height="50px"> <a href='https://harvard-iacs.github.io/2025-AC215/' target='_blank'><strong><font color="#A41034">AC215/E115: MLOps & LLMOps: Production AI Systems</font></strong></a></h1>

# **<font color="#A41034">Tutorial -  Agent Harness:  Formaggio Store</font>**

**Harvard University**<br/>
**Fall 2026**<br/>
**Instructor:** Pavlos Protopapas / Shivas Jayaram<br/>


<hr style="height:2pt">

## 📝 Make a Copy to Edit

This notebook is **view-only**. To edit it, follow these steps:

1. Click **File** > **Save a copy in Drive**.
2. Your own editable copy will open in a new tab.

Now you can modify and run the code freely!

# **Building an Agent Harness**


<center><img src="https://drive.google.com/uc?export=view&id=1oCldVicFepwdYbPHcD8Ui4VeyQVJYgut" height="400"><center>



In this notebook, we will brings together the concepts from the lecture: **context engineering**, **agent harnesses**, **skills**, **agent systems**, and **evaluation**. You will explore a cheese-shop assistant, add controlled workflows and human review, and inspect the evidence of what the system actually did.

As we build the harness we will review the code and compare behaviors to explain which parts are controlled by the model and which are enforced by the application.

# **Learning Objectives**


The goal: **help a customer plan a cheese tasting while preserving their constraints and controlling actions**.

The six executable acts follow the lecture concepts:

| Lecture concept | Tutorial activity | Key question |
|---|---|---|
| **1. Recap** | Act 1: an agent with instructions and tools | What can a tool-using agent propose? |
| **2. Context Engineering** | Act 2: compare basic and enriched context | What information should the model receive? |
| **3. Agent Harness** | Act 3: a controlled workflow; Act 4: planning, memory, compaction, and review | How does the runtime manage execution and enforce boundaries? |
| **4. Skills** | Act 5: load reusable instructions and resources when needed | How do skills guide work without granting permissions? |
| **5. Agent Systems** | Act 6: a harness invokes an ordering workflow | How do flexible planning and controlled execution work together? |
| **6. Evaluation** | Inspect traces, run repeated cases, and compare configurations | What evidence supports a claim that the system improved? |

> **Observability and governance run through every act.** Watch both the agent's response and the application's recorded decisions.

# **Understanding Agent Harness**

**What Is an Agent Harness?**<br>
---
A model on its own is just a function from text to text. It can read a prompt and produce a completion — and that is all it can do. It cannot call an API, read a file, remember yesterday, or check whether its own answer was correct.

An **agent harness** is the software wrapped around the model that turns that function into something that can actually do work:

> *"The software scaffolding around a model: the loop, tools, context management, and guardrails that turn raw intelligence into a working agent."* — Anthropic

**What we'll build**

Frameworks (SDKs) usually hide the harness. In this tutorial we do the opposite. We start with the smallest thing that works — an agent with instructions, a few tools, and a customer who wants cheese — and keep that agent fixed.

Then we add each concern (tracing, budgets, context, validation, and later approvals and memory) as a layer around it, and watch what each one adds to the record of what actually happened. By the end, you'll be able to read any agent framework and see the same skeleton underneath.

# Prerequisites

Let's make sure we have everything needed. We'll need to have an OpenAI API key for some parts of this tutorial.

Before we can start using the [OpenAI API](https://openai.com/blog/openai-api), we'll need to sign up for an API key from OpenAI. We can do this by visiting the [OpenAI API Keys](https://platform.openai.com/api-keys) page and creating a new API key.

In [ ]:
# @title Setup Agent Harness Repo
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
# @title Install Dependencies
import tomllib

if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

In [ ]:
# @title Imports
import os
import json
from getpass import getpass
from pathlib import Path

from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from openai import AsyncOpenAI

from acts.act1_agent import build_agent
from acts.act2_context import (build_act2, preview_context, summarize_context_run,
                               print_context_comparison, print_shipping_demo, shipping_policy_demo)
from acts.act3_workflow import build_act3
from acts.act4_harness import build_act4
from acts.act5_skills import build_act5
from acts.act6_composition import build_act6
from cli import (
    print_agent_result,
    print_cart,
    print_workflow_result,
    print_harness_result,
    print_composition_result,
)
from formaggio.config import MODEL, load_json
from formaggio.agents.context import (
    customer_ask, customer_message, instructions, load_scenario,
)
from formaggio.agents.harness import Harness
from formaggio.agents.layers import Trace, Budget, Context, CartCheck
from formaggio.agents.runtime import FUNCTION_LIMITS, MODEL_OPTIONS
from formaggio.agents.tools import build_tools
from formaggio.operations.observability import Recorder
from formaggio.shop.data_models import AgentReply
from formaggio.shop.store import Store
from formaggio.operations.chat_view import show_chat

import pandas as pd
from IPython.display import display

from uuid import uuid4
from formaggio.evaluation.notebook_lab import golden_cases
from formaggio.evaluation.live_evaluation import run_live_suite
from formaggio.operations.evaluation_view import show_evaluation


In [ ]:
# @title Setup OpenAI Key
# Using Google Colab's Secrets feature 🔑
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
import os
os.environ['OPENAI_API_KEY'] = api_key

### Shared settings

Choose `SCENARIO` for Act 1, `CONTEXT_SCENARIO` for both Act 2 modes, and `CONTEXT_OVERRIDE_SCENARIO` for the priority check. After changing it,
rerun the request/table cells and both acts. Later acts use their own named scenarios.
`USE_FIXTURES` changes Acts 3–6 only; Acts 1–2 always use live model calls.
The PA/NY counterexample is deliberately fixed and does not change your selected scenario.


In [ ]:
# @title Shared settings
SCENARIO = "standard"  # Act 1 baseline.
CONTEXT_SCENARIO = "personalized"  # Same request for both modes in Act 2.
CONTEXT_OVERRIDE_SCENARIO = "preference-override"  # Today overrides saved taste.
USE_FIXTURES = False  # Scripted responses in Acts 3–6 only; Acts 1–2 are live.
RUN_CONTEXT_EVALUATION = False  # Optional extra live calls at the end of Act 2.
CONTEXT_REPEATS = 3

# Evaluation lab: n different requests, each repeated m times.
EVAL_ACT = 6            # Choose 1–6; start with the complete agent system.
EVAL_N = 3              # First n applicable golden cases shown in the lab.
EVAL_M = 2              # Repeat each case m times (1–20).
EVAL_FIXTURES = True    # False = live model evaluation, with API usage.
EVAL_CONTEXT = "enriched"  # Act 2 only: "basic" or "enriched".

DB_PATH = Path("outputs/colab.sqlite")
OUTPUT_ROOT = Path("outputs/colab_artifacts")
MEMORY_PATH = Path("outputs/colab_memory.sqlite")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def ask_for_decision():
    while True:
        decision = input("Type approve or decline: ").strip().lower()
        if decision in {"approve", "decline"}:
            return decision
        print("Please enter approve or decline.")


async def review_order(ticket):
    print("\n👤 Manager review")
    print(customer_ask(ticket.request))
    print_cart(ticket.report)
    return ask_for_decision()


async def review_email(review):
    print("\n👤 Review this email draft")
    print(f"To: {review.draft.recipient}")
    print(f"Subject: {review.draft.subject}\n")
    print(review.draft.body)
    return ask_for_decision()


print(f"Model: {MODEL}")
print("Acts 3–6:", "scripted responses" if USE_FIXTURES else "live model")


# **Formaggio Store**

In this section we look at the mock data for this demo

In [ ]:
# @title Cheese Catalog

pd.set_option("display.width", 140)
catalog = pd.DataFrame(load_json("catalog.json"))
catalog["price_per_100g"] = catalog["cents_per_100g"].map(lambda c: f"${c / 100:.2f}")
catalog["allergens"] = catalog["allergens"].str.join(", ")
catalog[["product_id", "name", "country", "style", "funk", "raw_milk", "price_per_100g", "stock_g", "allergens"]]

In [ ]:
# @title Store Policies
policy = load_json("policies.json")
rules = {
    "Cheeses per tasting":            f'{policy["min_products"]}–{policy["max_products"]} distinct',
    "Grams per guest":                f'{policy["min_grams_per_guest"]}–{policy["max_grams_per_guest"]} g',
    "Minimum per cheese":             f'{policy["min_product_grams"]} g, in {policy["quantity_increment_g"]} g steps',
    "Manager approval above":         f'${policy["manager_threshold_cents"] / 100:.0f} cheese subtotal',
    "Raw-milk shipping blocked to":   ", ".join(policy["raw_milk_blocked_states"]),
    "Revisions allowed":              policy["max_revisions"],
    "Approved vendor recipients":     ", ".join(policy["allowed_vendor_recipients"]),
}
pd.Series(rules, name="value").to_frame()

In [ ]:
# @title Pairings
pairings = pd.DataFrame(load_json("pairings.json"))
pairings["styles"] = pairings["styles"].str.join(", ")
pairings["allergens"] = pairings["allergens"].map(lambda a: ", ".join(a) or "—")
pairings[["name", "kind", "styles", "allergens"]]

In [ ]:
# @title Customers
pd.DataFrame(load_json("customers.json"))

In [ ]:
# @title The Customer Request

request = load_scenario(SCENARIO)
display(customer_ask(request))
print()
print("Structured fields the application actually trusts:")
print(json.dumps(request.model_dump(mode="json"), indent=2))

In [ ]:
# @title Apply the selected request to the shelf

shelf_store = Store()
allergy_tags, allergy_issues = shelf_store.allergy_tags(request)
raw = pd.DataFrame(load_json("catalog.json"))
shipping_label = f"raw milk blocked → {request.state or 'destination unknown'}"
raw[shipping_label] = raw["raw_milk"] & (
    (request.state or "").strip().upper() in shelf_store.policy.raw_milk_blocked_states
)
raw["allergen conflict"] = raw["allergens"].map(lambda tags: bool(allergy_tags.intersection(tags)))
raw["insufficient stock"] = raw["stock_g"] < shelf_store.policy.min_product_grams
flags = [shipping_label, "allergen conflict", "insufficient stock"]
raw["passes product filters"] = ~raw[flags].any(axis=1)
requirements = []
for country in request.required_countries:
    label = f"from {country}"
    raw[label] = raw["country"].str.casefold() == country.casefold()
    requirements.append(label)
if request.required_min_funk is not None:
    label = f"funk ≥ {request.required_min_funk}"
    raw[label] = raw["funk"] >= request.required_min_funk
    requirements.append(label)

view = raw[["product_id", "country", "funk", "stock_g", *flags,
            "passes product filters", *requirements]]
display(view.style
    .map(lambda value: "background-color: #fde2e2" if value else "color: #888", subset=flags)
    .map(lambda value: "background-color: #dff5df" if value else "color: #888",
         subset=["passes product filters", *requirements]))
print("Country and funk requirements may be met by different cheeses.")
print("Passing these product filters does not validate the full cart or resolve missing request details.")
for issue in allergy_issues:
    print("Needs clarification:", issue.detail)


# **The Agent harness**

We now have a shop, its rules, and a customer with a request.

In this section we build the agent that serves that customer, and then wrap it in a harness
one layer at a time.

We keep the shop domain and constraints consistent while changing how the system organizes reasoning, tools, and control.

| Act | We add | So that the system can… |
|---|---|---|
| **1** | The agent, then four harness layers | Propose a cart and leave evidence of what it did |
| **2** | A richer context layer | See the right facts before it starts |
| **3** | A workflow that calls the agent as one step | Validate, revise, pause for a human, and place an order safely |
| **4** | Task list, memory, compaction, and approval | Plan longer work without losing the constraints or overstepping |
| **5** | Skills | Load procedures on demand, without gaining any new permissions |
| **6** | Composition: a harness that calls the Act 3 workflow | Combine flexible planning with controlled execution |

> As a rule of thumb: **the model proposes; the harness decides.**
Every time you see a result, ask which of the two produced it.


## Act 1 — Build an agent

**Question: Can the agent use tools to propose a cart, and can the harness independently check what it produced?**

An agent combines a model, instructions, and tools. Here, the model can look up cheeses, check stock, calculate prices, and propose a tasting.

The SDK handles the loop: call the model, run its requested tools, and return the results until it gives a final answer.

First, run the agent directly. It can propose a cart, but it cannot place an order.

In [ ]:
# @title Agent

store = Store()
request = load_scenario(SCENARIO)
print("👤 Customer ask:\n" + customer_ask(request))

async with AsyncOpenAI(timeout=45, max_retries=0) as api:
    client = OpenAIChatClient(
        model=MODEL,
        async_client=api,
        function_invocation_configuration=FUNCTION_LIMITS,
    )
    agent = Agent(
        client=client,
        name="FormaggioAssistant",
        instructions=instructions(store),
        tools=build_tools(store, request, None, None),
        default_options={**MODEL_OPTIONS, "response_format": AgentReply},
    )
    bare_response = await agent.run(customer_message(request))

print("\n💬 Agent's response:\n" + bare_response.value.message)


### Add checks around the agent

The harness runs the same agent definition with four named layers:

- **Trace:** show and record what happened.
- **Budget:** limit model calls, tool calls, and runtime.
- **Context:** choose the extra information supplied to the agent.
- **CartCheck:** independently check the final proposed cart.

`build_agent` contains the agent definition shown above, with hooks for these layers. Calling `.run(...)` starts a fresh run using the chosen scenario.

The cart check reports problems; it does not repair the cart or place an order.

In [ ]:
# @title Agent + Harness

act1 = (
    Harness(build_agent, model=MODEL, scenario=SCENARIO)
    .add(Trace())
    .add(Budget(model_calls=8, tool_calls=20, seconds=120))
    .add(Context("basic"))
    .add(CartCheck())
)

with Recorder(DB_PATH) as recorder:
    act1_result = await act1.run(recorder, progress=print)

print_agent_result(act1_result)

In [ ]:
# Display Customer View
show_chat(DB_PATH, act1_result["run_id"])

## Act 2 — Give the agent useful context

**Question: What useful information did we add, and did the agent use it appropriately?**

Our returning customer asks for a tasting without repeating their saved preferences:
**mild cheeses and nonalcoholic pairings**. `CONTEXT_SCENARIO` selects this example.
The model, tools, instructions and current request stay the same between modes.
Basic gets no saved profile; enriched gets this customer's profile and a filtered shelf.

Look at the actual cheese choices. Our classroom rubric calls funk **0–2** mild.
Enriched should choose mild cheeses; basic is not graded on a preference it never saw.
Read the final pairing suggestions yourself: the automated cheese score does not grade
pairing prose. A tool listing wine as an option is not the same as recommending wine.

A matching answer is evidence, not proof of causation. Both modes may match, and enriched
may fail. Keep those results. Repeat runs to measure consistency. `standard` remains an
optional baseline whose saved preferences overlap the current request.


In [ ]:
context_preview = preview_context(CONTEXT_SCENARIO)

print("📚 What changes before the first model call?")
print("Basic: the same shop rules, customer request, and tools; no extra context sources.")
for item in context_preview["enriched"]["sources"]:
    print(f"Enriched adds {item['source']}: {item['reason']}")

print("\nSaved preferences:", context_preview["enriched"]["sources"][-1]["content"].get("preferences", []))

print("\n🔎 Excluded from the enriched starting list:")
reason_labels = {
    "fictional_shipping_policy": "raw milk: blocked by the fictional destination policy",
    "allergen_conflict": "conflicts with the confirmed allergies",
    "insufficient_stock_for_minimum": "not enough stock for the minimum serving",
}
for product in context_preview["enriched"]["excluded_products"]:
    reasons = "; ".join(reason_labels.get(reason, reason) for reason in product["reasons"])
    print(f"- {product['product_id']}: {reasons}")


### See the harness apply the shipping policy

Context can help the model avoid a bad choice. **Application code must still check it.**
This fixed counterexample contains Comté, a raw-milk cheese. The same cart is rejected
for PA and accepted for NY by `Store.validate`, used by both the CartCheck layer and checkout.
This is separate from the agent's cart and makes no model call or order.

Act 2 reports the violation. Act 3 will show the workflow refusing this proposal,
requesting a revision, and placing only the corrected cart. The PA restriction is a
**fictional classroom stand-in for regulatory policy**, not an actual statement of law.


In [ ]:
print_shipping_demo()
shipping_reports = shipping_policy_demo()


### Run both context modes

Now compare the same selected scenario in both modes.


In [ ]:
context_results = {}
context_events = {}
context_comparison = []
for mode in ("basic", "enriched"):
    print(f"\n🧪 Running {mode} context")
    lesson = build_act2(scenario=CONTEXT_SCENARIO, mode=mode, model=MODEL)
    with Recorder(DB_PATH) as recorder:
        result = await lesson.run(recorder, progress=print)
        context_events[result["run_id"]] = recorder.timeline(result["run_id"])
        context_comparison.append(summarize_context_run(result, context_events[result["run_id"]]))
    context_results[mode] = result
    print_agent_result(result)

print_context_comparison(list(context_results.values()), context_events)
print("\nCost of each run (tokens summed across all calls):")
for row in context_comparison:
    print(f"{row['mode']}: {row['model_calls']} model calls; {row['tool_calls']} tool calls; "
          f"input tokens {row['input_tokens']}; output tokens {row['output_tokens']}")
print("None means usage was not recorded. More context is not automatically better.")


### Today takes priority

The same customer now asks for a cheese at funk 4 or higher. Run enriched again.
The context check should show today's request wins over the saved mild preference.
This makes one additional live run.


In [ ]:
override_lesson = build_act2(scenario=CONTEXT_OVERRIDE_SCENARIO, mode="enriched", model=MODEL)
with Recorder(DB_PATH) as recorder:
    override_result = await override_lesson.run(recorder, progress=print)
print_agent_result(override_result)


In [ ]:
for mode, result in context_results.items():
    show_chat(DB_PATH, result["run_id"], title=f"Act 2 · {mode} context")

### Optional: repeat the context comparison

One run is an example, not a reliable comparison. Set `RUN_CONTEXT_EVALUATION = True`
in the shared settings, then run the next cell to try each mode `CONTEXT_REPEATS` times.
It makes **additional live model calls**, including when `USE_FIXTURES = True`.
Leave the switch off to skip it. Every trial starts with fresh shop inventory.

Compare the fraction of carts that pass, the average number of calls, and input tokens.
If both modes perform equally well, the experiment has not shown a benefit from adding
context for this task. A few trials are still a small sample; record mixed or failed
results rather than choosing only the best examples.


In [ ]:
context_trials = []
if RUN_CONTEXT_EVALUATION:
    if type(CONTEXT_REPEATS) is not int or CONTEXT_REPEATS < 2:
        raise ValueError("Use at least two repeats for a comparison.")
    for repetition in range(CONTEXT_REPEATS):
        for mode in ("basic", "enriched"):
            lesson = build_act2(scenario=CONTEXT_SCENARIO, mode=mode, model=MODEL)
            try:
                with Recorder(DB_PATH) as recorder:
                    trial = await lesson.run(recorder)
                    row = summarize_context_run(trial, recorder.timeline(trial["run_id"]))
                row["error"] = None
            except Exception as error:
                row = {"mode": mode, "cart_check": "error", "error": str(error)}
            row["repetition"] = repetition + 1
            context_trials.append(row)
            print(f"Trial {repetition + 1} · {mode}: {row['cart_check']}")
            if row["error"]:
                print("  Error:", row["error"])

    print("\n📊 Repeated comparison")
    for mode in ("basic", "enriched"):
        rows = [row for row in context_trials if row["mode"] == mode]
        completed = [row for row in rows if not row["error"]]
        passed = sum(row["cart_check"] == "passed" for row in rows)
        print(f"{mode}: {passed}/{len(rows)} carts passed; {len(rows) - len(completed)} run errors")
        personalized = sum(row.get("preference_check") == "pass" for row in rows)
        print(f"  Applicable context checks passed: {personalized}/{len(rows)} (basic receives no saved preference)")
        for metric in ("model_calls", "tool_calls", "input_tokens", "output_tokens"):
            measured = [row[metric] for row in completed if row[metric] is not None]
            average = f"{sum(measured) / len(measured):.1f}" if measured else "unavailable"
            print(f"  Average {metric}: {average} ({len(measured)}/{len(rows)} trials measured)")
else:
    print("Optional repeated comparison skipped. Enable it in the shared settings when ready.")


## Act 3 — Put the agent inside a workflow

**Question: If the agent proposes an invalid cart, will the workflow reject it before checkout and require any necessary human approval?**

Now we move from proposing a cart to completing a simulated order. A proposing agent chooses the cheeses, while application code controls the steps:

**Confirm details → Propose cart → Validate → Select pairings → Check approval → Recheck and place**

An invalid cart goes back for revision, up to a limit. If manager approval is required, the workflow pauses for your decision. The model cannot approve its own order.

This example uses the manager-approval scenario. Inspect the cart before answering the prompt, then check the final workflow status.

In [ ]:
# @title Build Workflow

act3 = build_act3(
    scenario="manager-approval",
    model=MODEL,
    fixture=USE_FIXTURES,
)

with Recorder(DB_PATH) as recorder:
    act3_result = await act3.run(
        recorder,
        manager=review_order,
        progress=print,
    )

print_workflow_result(act3_result)

In [ ]:
show_chat(DB_PATH, act3_result["run_id"])

### Force a bad proposal to test the boundary

This scripted test makes the proposer suggest raw-milk Comté for PA.
Watch application code reject the first cart, request a revision, then check the
corrected cart before creating one mock receipt. A good final answer alone would
not demonstrate this rejection path. No model API is called.


In [ ]:
# Deliberately bad scripted proposal: demonstrate enforcement, without an API call.
pa_lesson = build_act3(scenario="pa-shipping", fixture=True, model=MODEL)
with Recorder(DB_PATH) as recorder:
    pa_result = await pa_lesson.run(recorder, progress=print)
print_workflow_result(pa_result)
print("Check: first attempt rejected for shipping; second attempt valid; only the corrected cart has a receipt.")


## Act 4 — Help an agent manage a longer task

**Question: Can the assistant plan a sourcing task while keeping vendor text untrusted and leaving the email decision to a human?**

This act introduces a planning agent. It checks an event menu, investigates a stock shortage, and prepares a supplier email for your review.

Its harness adds a task list, saved customer preferences, and a way to shorten long conversation history while keeping the current application state available. It also records activity and limits the work.

Watch how the agent updates its plan and uses the saved preference. Review the email before approving or declining it. Short runs may not need any history shortening.

In [ ]:
act4 = build_act4(
    scenario="event-shortage",
    model=MODEL,
    fixture=USE_FIXTURES,
    remember_preferences=("Prefer a concise supplier email.",),
    memory_path=MEMORY_PATH,
    output_root=OUTPUT_ROOT,
)

with Recorder(DB_PATH) as recorder:
    act4_result = await act4.run(
        recorder,
        reviewer=review_email,
        progress=print,
    )

print_harness_result(act4_result)
print("\n🧠 Saved preferences:", act4_result["preferences"])
print("📝 Final task list:")
for task in act4_result["tasks"]:
    print(task)

In [ ]:
show_chat(DB_PATH, act4_result["run_id"])

## Act 5 — Give the agent reusable skills

**Question: Can the assistant load the right procedure for a task without gaining extra permissions?**

A skill is a set of instructions and supporting files for a particular task. The agent can read those details when it needs them.

We reuse the planning agent and add skills. Compare two requests:

1. A simple stock question.
2. A tasting plan with serving order, pairings, and a quote.

Watch which skills and supporting files are loaded. The stock question should need little guidance; the tasting task can use the tasting-planning skill. A skill guides the work, but it cannot override application checks or grant approval.

In [ ]:
act5_results = {}

for scenario in ("stock-question", "tasting-plan"):
    print(f"\n🧪 Task: {scenario}")
    lesson = build_act5(
        scenario=scenario,
        model=MODEL,
        fixture=USE_FIXTURES,
        memory_path=MEMORY_PATH,
        output_root=OUTPUT_ROOT,
    )
    with Recorder(DB_PATH) as recorder:
        act5_results[scenario] = await lesson.run(recorder, progress=print)
    print_harness_result(act5_results[scenario])

print("\n📚 Skills used")
for scenario, result in act5_results.items():
    print(f"{scenario}: {', '.join(result['skills_loaded']) or 'none'}")

In [ ]:
for scenario, result in act5_results.items():
    show_chat(DB_PATH, result["run_id"], title=f"Act 5 · {scenario}")

## Act 6 — Bring the pieces together

**Question: Can a flexible assistant use the governed workflow and turn its verified result into a complete tasting plan?**

A customer-facing assistant now uses the ordering workflow as one of its tools. The workflow has its own proposing agent and still controls validation, approvals, and simulated checkout.

The assistant can then use a skill to turn the accepted menu into a tasting plan.

This connects the lecture's main ideas: useful context, agents with tools, harness checks, reusable skills, and an agent system with separate responsibilities.

To evaluate the result, check two things separately: **what happened to the order**, and **whether the requested tasting plan was delivered**. A confident final message alone is not proof of success.

In [ ]:
act6 = build_act6(
    scenario="standard",
    model=MODEL,
    fixture=USE_FIXTURES,
)

with Recorder(DB_PATH) as recorder:
    act6_result = await act6.run(
        recorder,
        manager=review_order,
        progress=print,
    )
    act6_events = recorder.timeline(act6_result["run_id"])

print_composition_result(act6_result)

print("\n🔍 Was the requested tasting plan delivered?")
delivery_checks = [
    event["payload"] for event in act6_events
    if event["event_type"] == "delivery.checked"
]
if delivery_checks:
    for check in delivery_checks:
        print("Ready:", check["ready"])
        for error in check.get("errors", []):
            print("-", error)
else:
    print("No delivery check was recorded. Inspect the run status above.")

In [ ]:
show_chat(DB_PATH, act6_result["run_id"])

## Evaluation lab — Test the agents across requests and repeated runs

We have watched individual runs. Now ask: **does the system do the right thing across
several requests, and does it keep doing the right thing when we repeat them?**

A **golden case** is a mock request with an expected outcome. We check properties,
such as “ask for missing details” or “place one valid mock order,” rather than requiring
one exact answer or cheese selection. A refusal or clarification can be a success.

Use the shared settings at the top:

- `EVAL_ACT`: which act to evaluate; start with Act 6, which combines the assistant and workflow.
- `EVAL_N`: number of different applicable cases, selected from the list below.
- `EVAL_M`: repetitions of each case. There are **n × m scheduled runs**.
- `EVAL_FIXTURES`: start with `True` to learn the evaluation process using scripted responses.
  Set it to `False` to measure the live model. This setting is independent of `USE_FIXTURES`.
- `EVAL_CONTEXT`: select basic or enriched context when evaluating Act 2.

With the defaults, three requests run twice: **six scripted runs, with no API calls**.
Scripted consistency does not demonstrate model robustness. Live runs make multiple
model calls per request and incur API usage. To study one request repeatedly, use
`EVAL_N = 1` and increase `EVAL_M`.

Every run starts with fresh inventory, session state, and preference memory. Review
choices are explicit test decisions supplied by each case, even in live mode; the
model never acts as its own manager. No real purchases or emails are sent.

### 1. Inspect the golden requests and expected outcomes

The readable requests below summarize the confirmed scenario data. The agents receive
that data through their normal act-specific inputs; exact model messages remain in
the trace. Changing this display text does not change the test. Case definitions live
in `data/agent_evaluation_cases.json`, with request fields in `data/scenarios.json`.


In [ ]:
available_golden_cases = golden_cases(EVAL_ACT)
if type(EVAL_N) is not int or not 1 <= EVAL_N <= len(available_golden_cases):
    raise ValueError(f"For Act {EVAL_ACT}, choose EVAL_N from 1 to {len(available_golden_cases)}.")
if type(EVAL_M) is not int or not 1 <= EVAL_M <= 20:
    raise ValueError("Choose EVAL_M from 1 to 20; use at least 2 to explore repeatability.")
if EVAL_ACT == 2 and EVAL_CONTEXT not in {"basic", "enriched"}:
    raise ValueError("For this lab, choose one context mode: basic or enriched.")

selected_golden_cases = available_golden_cases[:EVAL_N]
for index, case in enumerate(available_golden_cases, 1):
    print(f"\n{'✅ Selected' if index <= EVAL_N else '◻️ Available'} · {index}. {case['case_id']}")
    print("👤 Request:", case["request"])
    print("🎯 Expected:", case["expected"])
    print(f"Review decision if requested: {case['review_decision']}; vendor document: {case['document']}")

print(f"\nScheduled: {EVAL_N} requests × {EVAL_M} repeats = {EVAL_N * EVAL_M} runs")
print("Mode:", "SCRIPTED — tests the evaluation machinery" if EVAL_FIXTURES else "LIVE — measures model behavior")

evaluation_selection = (EVAL_ACT, EVAL_N, EVAL_M, EVAL_CONTEXT)


### 2. Run the selected cases

Each execution gets a new experiment label, so rerunning the cell preserves the previous
batch. The runner uses the real act implementations and the same checks as the CLI.

It records successful runs, failed checks, execution errors, and any missing scheduled
runs. After three consecutive execution errors it stops the batch; missing runs still
count in the pass-rate denominator. Manager-review cases also report whether review
was actually reached—a cheap cart can bypass that branch without testing it.


In [ ]:
# Rerun the selection cell after changing the evaluation settings.
if evaluation_selection != (EVAL_ACT, EVAL_N, EVAL_M, EVAL_CONTEXT):
    raise ValueError("Evaluation settings changed; rerun the golden-case selection cell first.")
evaluation_label = f"colab-act{EVAL_ACT}-{uuid4().hex[:12]}"


def evaluation_progress(message):
    # Full model/tool traces are saved; keep the notebook's progress output short.
    if message.startswith(("FIXTURE", "LIVE", "Case ")):
        print(message)


with Recorder(DB_PATH) as recorder:
    evaluation_report = await run_live_suite(
        recorder,
        evaluation_label,
        act=EVAL_ACT,
        case_ids=[case["case_id"] for case in selected_golden_cases],
        repeats=EVAL_M,
        context=EVAL_CONTEXT if EVAL_ACT == 2 else None,
        model=MODEL,
        fixture=EVAL_FIXTURES,
        output_root=OUTPUT_ROOT,
        progress=evaluation_progress,
    )
    evaluation_export = {
        "report": evaluation_report,
        "traces": {run["run_id"]: recorder.timeline(run["run_id"])
                   for run in evaluation_report["runs"]},
    }

evaluation_json = OUTPUT_ROOT / f"{evaluation_label}.json"
evaluation_json.write_text(json.dumps(evaluation_export, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nSaved complete results and recorded events: {evaluation_json}")
print(f"Experiment: {evaluation_label} · database: {DB_PATH}")


### 3. Read the scorecard and inspect every run

The view below reads the saved batch; rerunning this display cell makes no model calls.

| Metric | What it tells us |
|---|---|
| **Passed / scheduled** | How often all applicable automated checks passed; errors and missing runs remain in the denominator. |
| **Outcome agreement** | How often the most common completed outcome occurred across the scheduled repeats. Consistent behavior can still be wrong. |
| **Distinct results, responses, and tool paths** | How much the outputs and execution paths vary. Different valid menus or wording are allowed. |
| **Model calls and tokens** | How much model work each run required, including inner and outer agents. Missing measurements are shown as unavailable. |
| **Latency** | Recorded end-to-end run time, including tools and scripted review handling. |
| **Untested paths** | Checks such as manager review that the run never reached. |

Expand any run to inspect **expected vs. observed checks, the final output, and the full
sequence of recorded events**. Large payload previews may be shortened; the JSON export
contains the complete recorded events. In Colab, use the Files sidebar to download the
printed JSON file or the SQLite database for later inspection.

There is no automatic prose-quality score. Read at least one passing and one failing
run, when available. Check clarity, accurate action status, and whether the response
actually helps the customer. A small batch estimates behavior; it does not prove robustness.


In [ ]:
show_evaluation(DB_PATH, evaluation_report)


### 4. Explain what you learned

1. Which request passed least often? Expand its runs and identify the failing check.
2. Did the same request produce different valid menus, different outcomes, or errors?
3. Which case used the most tokens or took longest? What does its trace explain?
4. Did a manager-review case actually exercise approval or decline?
5. Record the mode, model, case IDs, number of repeats, and experiment label with your conclusion.

Start with scripted runs to understand the scorecard, then switch to live mode for a
small batch. To compare two acts, use case IDs that apply to both and account for their
different responsibilities: Act 1 proposes a cart; Act 6 can place a mock order and deliver
a plan. Matching wording is not the definition of success.


## Evaluation and reflection — What did the harness actually do?

A convincing answer, a valid cart, an approved order, and a useful tasting plan are
separate results. Check the recorded evidence for each one. `Ready: True` checks the
required plan structure and menu coverage; still read the response for clarity.

| Act | One question to answer from your run |
|---|---|
| 1 — Agent and harness | Which check came from application code rather than the model? |
| 2 — Context | What useful information was added, and did the agent use it appropriately? |
| 3 — Workflow | What happens if you type `decline` at the approval step? |
| 4 — Memory and review | What was remembered, and was the email saved or actually sent? |
| 5 — Skills | Why did the tasting task load a skill while the stock question did not? |
| 6 — Agent system | Did checkout succeed, did delivery pass, and is the final plan easy to use? |

**Try one failure:** rerun the Act 3 or Act 4 cell and decline the review. Inspect the
chat view for the recorded decision and confirm that the corresponding action did not
happen. These reruns use live models unless you enable scripted responses for Acts 3–6.

For a small measurement exercise, use the optional repeated comparison in Act 2.
Write three sentences: what changed, what the recorded results show, and what they
do not yet prove. Equal results are a valid conclusion.
